# Kocur 3D boxes on I-24, scored against the annotated 3D boxes

Runs the **authors' code and weights** for the Kocur 3D-box method (YOLOv6-3D: Macko, Gajdošech & Kocur 2025,
the YOLOv6 version of Kocur & Ftáčnik 2020; repo `gajdosech2/Vehicle-Speed-Estimation-YOLOv6-3D`) on I-24 3D
videos, then compares every box with the dataset's annotated cuboids:

- **frame offset** between video and annotations (searched automatically),
- **box accuracy** on the road plane: footprint-center error, error of the ground point used for speed,
  length / width / height error, bird's-eye IoU, recall and precision,
- **speed** per vehicle from that ground point vs the annotated positions (same frames, same timestamps).

The vanishing points come from the dataset's `hg.json` (dataset calibration, not auto-calibration), and the
road mask is the hull of the annotated vehicle footprints. Only cameras that look along the road are used: the
method's warp needs the VP2-VP3 line to stay outside the image, which fails for the side-looking cameras.

Use a GPU runtime (Runtime → Change runtime type → T4). Results go to `Final-Project-CHULA/runs/kocur_i24/`.

In [ ]:
# ---- settings ----
SCENE = 'scene1'
CAMERAS = ['p2c1', 'p2c6', 'p3c1', 'p3c5']   # along-road cameras where the Kocur warp is valid
MAX_FRAMES = None        # None = whole video (~3400 frames); e.g. 900 for a quick try
CONF = 0.65              # authors' default detection threshold
BRANCH = 'claude/project-thread-rwrrf5'

In [ ]:
import os
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = next((c for c in ['/content/drive/MyDrive/Final-Project-CHULA',
                            '/content/drive/Shareddrives/Final-Project-CHULA'] if os.path.isdir(c)), None)
assert PROJECT, 'Add a shortcut to Final-Project-CHULA in MyDrive, or set PROJECT manually'

DATA_DIR, VIDEO_DIR = None, None
for root, dirs, files in os.walk(PROJECT):
    if DATA_DIR is None and {'obj', 'ts', 'hg'}.issubset(dirs):
        DATA_DIR = root
    if VIDEO_DIR is None and os.path.basename(root) == SCENE and any(f.endswith('.mp4') for f in files):
        VIDEO_DIR = root
    if DATA_DIR and VIDEO_DIR:
        break
print('DATA_DIR  =', DATA_DIR)
print('VIDEO_DIR =', VIDEO_DIR)
assert DATA_DIR and VIDEO_DIR, 'I-24 data not found under Final-Project-CHULA'

In [ ]:
%cd /content
!rm -rf capcapcar-speed kocur
!git clone -q -b {BRANCH} https://github.com/WalkerCze3/capcapcar-speed.git
!git clone -q --depth 1 https://github.com/gajdosech2/Vehicle-Speed-Estimation-YOLOv6-3D.git kocur
!pip install -q addict pycocotools shapely
%cd /content/capcapcar-speed

## Run + score each camera

About 3-6 minutes per camera on a T4 (decoding the 4K video is the slow part).

In [ ]:
import json, shutil, subprocess

def sh(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'Warning' not in line:
            print(line, end='', flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

OUT_DRIVE = f'{PROJECT}/runs/kocur_i24'
summaries = {}
for cam in CAMERAS:
    video = f'{VIDEO_DIR}/{cam}.mp4'
    run = f'runs/kocur_i24/{SCENE}_{cam}'
    print(f'===== {cam} =====')
    cmd = ['python', 'baselines/kocur_i24/run_kocur_i24.py', '--kocur-repo', '/content/kocur', '--video', video,
           '--data-dir', DATA_DIR, '--scene', SCENE, '--camera', cam, '--out', run, '--conf', str(CONF)]
    if MAX_FRAMES:
        cmd += ['--max-frames', str(MAX_FRAMES)]
    try:
        sh(cmd)
        sh(['python', 'baselines/kocur_i24/eval_kocur_i24.py', '--run-dir', run, '--data-dir', DATA_DIR,
            '--video', video])
        summaries[cam] = json.load(open(f'{run}/summary.json'))
    except Exception as e:  # keep going with the other cameras; the log above shows why
        print(f'!! {cam} failed: {e}')
    if os.path.isdir(run):
        dst = f'{OUT_DRIVE}/{SCENE}_{cam}'
        shutil.copytree(run, dst, dirs_exist_ok=True)
        print('saved', dst)

## Results

In [ ]:
import pandas as pd
rows = []
for cam, s in summaries.items():
    r = {'camera': cam, 'offset': s['offset'], 'iou2d': s['mean_best_2d_iou']['best'],
         'matched': s['matched'], 'recall': s['recall'], 'precision': s['precision']}
    for k in ['center_err_m', 'ground_point_err_m', 'length_abs_err_m', 'width_abs_err_m', 'height_abs_err_m', 'bev_iou']:
        r[k + ' (median)'] = s.get(k, {}).get('median')
    sp = s.get('speed', {})
    r.update({'vehicles': sp.get('vehicles'), 'speed MAE km/h': sp.get('mae_kmh'), 'speed bias km/h': sp.get('bias_kmh')})
    rows.append(r)
table = pd.DataFrame(rows)
table.to_csv(f'{OUT_DRIVE}/{SCENE}_summary.csv', index=False)
table.round(3)

In [ ]:
# green = annotated cuboid, orange = Kocur box, yellow dot = Kocur ground point used for speed
import glob, cv2
import matplotlib.pyplot as plt
for cam in summaries:
    for path in sorted(glob.glob(f'runs/kocur_i24/{SCENE}_{cam}/overlay_*.jpg'))[:2]:
        img = cv2.imread(path)[:, :, ::-1]
        plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.title(f'{cam} {os.path.basename(path)}')
        plt.show()